# Lekcija 09 - Uzorak dizajna metakognicije


## Postavljanje

Ovaj bilježnik prikazuje dizajnerski obrazac metakognicije koristeći Microsoft Agent Framework.

**Preduvjeti:**
- Implementacija Azure OpenAI-a konfigurirana putem varijabli okruženja
- Azure CLI prijavljen (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Što je metakognicija?

Metakognicija je **razmišljanje o razmišljanju**. U kontekstu agenata umjetne inteligencije, to znači izgraditi agente koji mogu:

- **Reflektirati** o vlastitim ishodima i procesu rezoniranja
- **Otkrivati pogreške** i oporaviti se na primjeren način umjesto da neprimjetno zakažu
- **Procijeniti** jesu li njihovi odgovori potpuni i korisni
- **Prilagoditi** svoju strategiju kada početni pristup ne uspije (npr. povratak na rezervni sustav)

Metakognitivni agent ne samo odgovara na pitanja — on nadzire vlastitu izvedbu i prilagođava se u hodu.


## Primarni i rezervni alati

Čest metakognitivni obrazac je **rezervna strategija**. Agent prvo pokušava s primarnim alatom; ako ne uspije (npr. greška 404), agent prepozna neuspjeh i transparentno prijeđe na rezervni alat.

Ovo odražava stvarne sustave u kojima primarne usluge mogu biti nedostupne i agenti moraju samostalno dijagnosticirati problem prije nego što odaberu alternativni put.

Ispod definiramo dva alata za pretraživanje letova:
- **Primarni** — pokriva Paris, Tokyo i Barcelona
- **Rezervni** — pokriva Berlin, Sydney i New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent u nastavku koji se samoreflektira s oporavkom od pogrešaka

Agent u nastavku dobiva upute da najprije pokuša primarni sustav leta, prepozna kvarove i transparentno se prebaci na rezervni sustav. Nakon svakog odgovora kratko se samoevaluira je li u potpunosti odgovorio na korisnikovo pitanje.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Uzorak samoocjenjivanja

Drugi aspekt metakognicije je **samoocjenjivanje**: zaseban agent (ili isti agent u drugom prolazu) pregledava odgovor radi potpunosti, točnosti i korisnosti.

U nastavku stvaramo agenta `ResponseEvaluator` koji ocjenjuje odgovore agenta za putovanja prema trima dimenzijama.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sažetak

U ovoj lekciji naučili ste kako izgraditi **metakognitivne agente** koristeći Microsoft Agent Framework:

- **Samorefleksija**: Agenti koji nadgledaju vlastito zaključivanje i transparentno komuniciraju što se dogodilo.
- **Oporavak od pogrešaka s rezervnim opcijama**: Uzorak primarnog + rezervnog alata gdje agent otkriva pogreške (npr. pogreške 404) i automatski pokušava alternativni izvor.
- **Samoocjenjivanje**: Poseban evaluacijski agent koji ocjenjuje odgovore prema potpunosti, točnosti i korisnosti.

Ovi obrasci čine agente robusnijima, transparentnijima i pouzdanijima — ključnim svojstvima za produkcijsko uvođenje.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Odricanje odgovornosti:
Ovaj je dokument preveden pomoću AI usluge za prevođenje Co-op Translator (https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati mjerodavnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakve nesporazume ili pogrešna tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
